# Random rollout on CartPole

This notebook shows the smallest end-to-end loop: build a **vector** environment from `EnvConfig`, sample random actions, and inspect the plain dict records returned each step.

See [docs/guide.md](../docs/guide.md) for the full step contract.

## Imports

`EnvConfig` accepts any Gymnasium environment id through `id`. `make_vector_env` wraps Gymnasium and formats steps as `(result, metrics)`.

In [ ]:
from tensordict import TensorDict as TD
from mouse_envs import EnvConfig, make_vector_env

## Configure the environment

- `num_envs=1` runs one CartPole instance.
- `max_episode_steps` caps episode length (truncation).
- `seed` fixes RNG for reproducibility.

In [ ]:
cfg = EnvConfig(
    id="CartPole-v1",
    seed=0,
    num_envs=1,
    max_episode_steps=500,
)
env = make_vector_env(cfg)
env.name

## Rollout loop

Environment names are available by vector index as `env.names`; `env.name` returns the first one for single-env use.

Each `env.step` returns:

- **`result[i]`** — `time`, `observation` (dict: `discrete` / `continuous` / `image`), `reward`, `done`, `episode_index`, `reward_episodic`, optional `q_star` / `ns_params`
- **`metrics[i]`** — `episode_cum_reward` and `episode_length` lists for finishes on this step (empty if none)

Each **`actions[i]["action"]`** passed to `step()` is also a dict — `discrete` or `continuous`, matching the env's action space.

The **first** `step` call performs an internal reset (actions are ignored); see the guide for the initial frame (`time == 0`).

In [ ]:
for step in range(1000):
    actions = env.sample_random_actions()
    result, metrics = env.step(actions)
    print(result[0])

## Cleanup

In [ ]:
env.close()